In [1]:
import os, random, torch
import pandas as pd
import numpy as np
from pathlib import Path
from dataclasses import dataclass
from torch.utils.data import Dataset, DataLoader

In [2]:
import os
import numpy as np
import pandas as pd

FS = 64000
CH = "CH17"
COND1, COND2, COMP = "20Hz", "0kN", "leftaxlebox"

# TaskA
FOLDERS = {
    "la2_or": "M0_G0_LA2_RA0",
    "la3_bf": "M0_G0_LA3_RA0",
    "mix":    "M0_G0_LA2+LA3_RA0",
}

def zscore(x):
    x = x.astype(np.float32)
    return (x - x.mean()) / (x.std() + 1e-8)

def list_csvs(root, folder):
    base = os.path.join(root, folder)
    out = []
    if not os.path.isdir(base): return out
    for sample in sorted(os.listdir(base)):
        sp = os.path.join(base, sample)
        if not os.path.isdir(sp): continue
        for fn in os.listdir(sp):
            if not fn.endswith(".csv"): continue
            if COND1 not in fn or COND2 not in fn or COMP not in fn: continue
            out.append((sample, os.path.join(sp, fn), fn))
    return out

def make_slices_from_file(csv_path, win, stride, max_slices=None):
    sig = pd.read_csv(csv_path, usecols=[CH])[CH].values.astype(np.float32)
    sig = zscore(sig)
    n = len(sig)
    slices, starts = [], []
    for s in range(0, n - win + 1, stride):
        slices.append(sig[s:s+win])
        starts.append(s)
        if max_slices and len(slices) >= max_slices:
            break
    if len(slices) == 0:
        return None, None
    return np.stack(slices, 0), np.array(starts, dtype=np.int32)

def build_dataset(root, out_npz="TaskA_BSS_dataset.npz",
                  win=8192, stride=2048,
                  max_per_file=80, max_total_per_class=3000):
    # 统一采样：每个类最多 max_total_per_class 个切片，且每个文件最多 max_per_file 个
    data = {"la2_or": [], "la3_bf": [], "mix": []}
    meta = {"la2_or": [], "la3_bf": [], "mix": []}

    for key, folder in FOLDERS.items():
        files = list_csvs(root, folder)
        cnt = 0
        for sample, fp, fn in files:
            xs, starts = make_slices_from_file(fp, win, stride, max_slices=max_per_file)
            if xs is None: 
                continue
            for i in range(xs.shape[0]):
                data[key].append(xs[i])
                meta[key].append((sample, fn, int(starts[i])))
                cnt += 1
                if cnt >= max_total_per_class:
                    break
            if cnt >= max_total_per_class:
                break

        data[key] = np.stack(data[key], 0).astype(np.float32) if len(data[key]) else np.zeros((0, win), np.float32)
        meta[key] = np.array(meta[key], dtype=object)
        print(key, "segments =", len(meta[key]))

    np.savez_compressed(out_npz,
                        fs=FS, win=win, stride=stride,
                        x_or=data["la2_or"], m_or=meta["la2_or"],
                        x_bf=data["la3_bf"], m_bf=meta["la3_bf"],
                        x_mix=data["mix"],    m_mix=meta["mix"])
    print("saved:", out_npz)

if __name__ == "__main__":
    ROOT = r"E:\BaiduNetdiskDownload\BJTU-RAO Bogie Datasets\Data\BJTU_RAO_Bogie_Datasets\BJTU_RAO_Bogie_Datasets"
    build_dataset(ROOT)

la2_or segments = 240
la3_bf segments = 240
mix segments = 240
saved: TaskA_BSS_dataset.npz


In [3]:
import numpy as np
import matplotlib.pyplot as plt
from scipy.signal import hilbert
from matplotlib.backends.backend_pdf import PdfPages

def get_env_spec(sig, fs):
    """计算包络谱核心逻辑"""
    env = np.abs(hilbert(sig))
    env -= np.mean(env) # 去直流
    n = len(env)
    p = np.abs(np.fft.fft(env)) / n
    f = np.fft.fftfreq(n, 1/fs)
    return f, p

def plot_all_to_pdf(npz_path="TaskA_BSS_dataset.npz", rows=5, cols=2):
    d = np.load(npz_path, allow_pickle=True)
    fs = int(d['fs'])
    per_page = rows * cols # 每页画10张

    categories = [
        ('x_or', 'OR_Inspection.pdf', 23.4, 'tab:red'),
        ('x_bf', 'BF_Inspection.pdf', 9.7, 'tab:green'),
        ('x_mix', 'Mix_Inspection.pdf', [9.7, 23.4], 'tab:blue')
    ]

    for key, pdf_name, fcf, color in categories:
        signals = d[key]
        num_sigs = len(signals)
        if num_sigs == 0: continue
        
        print(f"正在生成 {pdf_name}，共 {num_sigs} 个切片...")
        
        with PdfPages(pdf_name) as pdf:
            for i in range(0, num_sigs, per_page):
                fig, axes = plt.subplots(rows, cols, figsize=(12, 18))
                axes = axes.flatten()
                
                for j in range(per_page):
                    idx = i + j
                    if idx >= num_sigs:
                        axes[j].axis('off')
                        continue
                    
                    f, p = get_env_spec(signals[idx], fs)
                    # 仅画 0-200Hz
                    mask = (f >= 0) & (f <= 200)
                    axes[j].plot(f[mask], p[mask], color=color, lw=0.8)
                    
                    # 标注特征频率
                    fcfs = [fcf] if isinstance(fcf, (float, int)) else fcf
                    for f_val in fcfs:
                        axes[j].axvline(f_val, color='black', ls='--', alpha=0.3)
                    
                    axes[j].set_title(f"Slice {idx}", fontsize=9)
                    axes[j].tick_params(labelsize=7)

                plt.tight_layout()
                pdf.savefig(fig)
                plt.close(fig)
                
                if (i // per_page) % 10 == 0:
                    print(f"  已处理 {idx+1}/{num_sigs}...")

    print("✅ 所有 PDF 已生成，请在文件夹中查看。")

if __name__ == "__main__":
    plot_all_to_pdf()

正在生成 OR_Inspection.pdf，共 240 个切片...
  已处理 10/240...
  已处理 110/240...
  已处理 210/240...
正在生成 BF_Inspection.pdf，共 240 个切片...
  已处理 10/240...
  已处理 110/240...
  已处理 210/240...
正在生成 Mix_Inspection.pdf，共 240 个切片...
  已处理 10/240...
  已处理 110/240...
  已处理 210/240...
✅ 所有 PDF 已生成，请在文件夹中查看。
